# **3 ВАРИАНТ** Танюшкин А.Л.

# Лабораторная работа № 2. Работа с mlflow
    Цель лабораторной работы – получение навыков логирования своей работы в mlflow.

1.	Использовать задание из вашего варианта первой лабораторной работы

✓

2.	Настроить окружение проекта локально на комьютере:

a.	Создать отдельную директорию под эту лабораторку

b.	Установить зависимости в отдельное виртуальное окружение для этой лабы

c.	Обязательно установить mlflow как зависимость

d.	Прописать зависимости лабораторной в файле requirements.txt (или использовать poetry)

e.	Скопировать ваш ноутбук из первой лабы в папку с лабораторной работой


a.	Создать отдельную директорию под эту лабораторку

✓

b.	Установить зависимости в отдельное виртуальное окружение для этой лабы

✓

c.	Обязательно установить mlflow как зависимость

✓

d.	Прописать зависимости лабораторной в файле requirements.txt (или использовать poetry)

✓

e.	Скопировать ваш ноутбук из первой лабы в папку с лабораторной работой

✓

3.	Далее необходимо запустить Mlflow Server

✓

4.	Ноутбук из своей первой лабораторной модифицируете по следующим правилам:

a.	Добавляете подключение к mlflow серверу

b.	Включаете автоматическое логирование экспериментов keras

i.	Логировать сами модели надо

ii.	Логировать датасеты не надо

c.	Делаете один прогон обучения с набором параметров – у вас появится новый ран в mlflow

d.	Копируете ID рана, который вы получили от mlflow. Вновь его запускаете и добавляете метрики качества, которые вы оцениваете на тестовом датасете. (пример см. здесь)

e.	Далее меняете набор параметров или архитектуру модели

f.	Повторяйте шаги C-E минимум 3 раза

g.	В результате вы получите минимум три рана в вашем сервере mlflow


In [10]:
import os
import mlflow
#import mlflow.tensorflow
#from mlflow.tensorflow import MlflowCallback
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from tensorflow import keras
from tensorflow.keras.layers import Dense, Dropout # type: ignore
import keras_tuner as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow

In [11]:
df = pd.read_csv('data.csv')

In [12]:
df.head(1)

,Unnamed: 0,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,state,...,mosque_count_5000,leisure_count_5000,sport_count_5000,market_count_5000,ecology_excellent,ecology_good,ecology_no data,ecology_poor,ecology_satisfactory,price_doc
0,0,-0.294873,-0.2052,-0.690611,0.128662,-0.212332,-0.004459,-0.064771,-0.083784,0.393775,...,0.915176,-0.420245,-0.017208,-0.406425,-0.385252,1.80206,-0.579283,-0.59758,-0.370907,-0.266324


In [13]:
X = df.drop('price_doc', axis=1)
y = df['price_doc']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=5
)

In [15]:
os.environ['USER'] = 'Aleggg'

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment('experiment_with_mlflow_lab2')

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1762650347971, experiment_id='1', last_update_time=1762650347971, lifecycle_stage='active', name='experiment_with_mlflow_lab2', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [16]:
mlflow.keras.autolog(log_models=True,log_datasets=False)

In [17]:
def calculate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mse)
    
    return {
        "test_mse": mse,
        "test_mae": mae, 
        "test_r2": r2,
        "test_rmse": rmse
    }

In [18]:
def create_model(hidden_layers, units, dropout_rate, learning_rate):
    model = keras.Sequential()
    
    model.add(Dense(units, activation='relu', input_shape=(X_train.shape[1],)))
    model.add(Dropout(dropout_rate))
    
    for i in range(hidden_layers - 1):
        model.add(Dense(units // (2 ** (i + 1)), activation='relu'))
        model.add(Dropout(dropout_rate))
    
    model.add(Dense(1))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'])
    
    return model

In [19]:
def first_run(run_name='first_run'):
    with mlflow.start_run(run_name=run_name):
        params_1 = {
            'hidden_layers': 3,
            'units': 128,
            'dropout_rate': 0.3,
            'learning_rate': 0.001,
            'epochs': 50,
            'batch_size': 32
            }
        
        mlflow.log_params(params_1)

        model_1 = create_model(
            hidden_layers=params_1['hidden_layers'],
            units=params_1['units'],
            dropout_rate=params_1['dropout_rate'],
            learning_rate=params_1['learning_rate']
        )

        model_1.fit(X_train, y_train, epochs=params_1['epochs'],
                    batch_size=params_1['batch_size'], 
                    validation_data=(X_test, y_test),
                    verbose=1)
        
        y_pred = model_1.predict(X_test).flatten()

        metrics = calculate_metrics(y_test, y_pred)
        mlflow.log_metrics(metrics)

In [20]:
def second_run(run_name='second_run'):
    with mlflow.start_run(run_name=run_name):
        params_2 = {
            'hidden_layers': 4,
            'units': 256,
            'dropout_rate': 0.2,
            'learning_rate': 0.0005,
            'epochs': 30,
            'batch_size': 64
            }
        
        mlflow.log_params(params_2)

        model_2 = create_model(
            hidden_layers=params_2['hidden_layers'],
            units=params_2['units'],
            dropout_rate=params_2['dropout_rate'],
            learning_rate=params_2['learning_rate']
        )

        model_2.fit(X_train, y_train, epochs=params_2['epochs'],
                    batch_size=params_2['batch_size'], 
                    validation_data=(X_test, y_test),
                    verbose=1)
        
        y_pred = model_2.predict(X_test).flatten()

        metrics = calculate_metrics(y_test, y_pred)
        mlflow.log_metrics(metrics)

In [21]:
def third_run(run_name='third_run'):
    with mlflow.start_run(run_name=run_name):
        params_3 = {
            'hidden_layers': 4,
            'units': 512,
            'dropout_rate': 0.6,
            'learning_rate': 0.001,
            'epochs': 100,
            'batch_size': 64
            }
        
        mlflow.log_params(params_3)

        model_3 = create_model(
            hidden_layers=params_3['hidden_layers'],
            units=params_3['units'],
            dropout_rate=params_3['dropout_rate'],
            learning_rate=params_3['learning_rate']
        )

        model_3.fit(X_train, y_train, epochs=params_3['epochs'],
                    batch_size=params_3['batch_size'], 
                    validation_data=(X_test, y_test),
                    verbose=1)
        
        y_pred = model_3.predict(X_test).flatten()

        metrics = calculate_metrics(y_test, y_pred)
        mlflow.log_metrics(metrics)

In [22]:
first_run()

c:\Users\Aleggg\Documents\code\.venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 30067.7812 - mae: 67.8564 - val_loss: 1.0075 - val_mae: 0.6091
Epoch 2/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 492.5400 - mae: 7.0265 - val_loss: 1.0077 - val_mae: 0.6077
Epoch 3/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 164.9518 - mae: 3.0136 - val_loss: 1.0077 - val_mae: 0.6077
Epoch 4/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 53.6910 - mae: 1.6303 - val_loss: 1.0076 - val_mae: 0.6083
Epoch 5/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 34.7879 - mae: 1.2251 - val_loss: 1.0079 - val_mae: 0.6074
Epoch 6/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 24.8951 - mae: 0.9725 - val_loss: 1.0081 - val_mae: 0.6062
Epoch 7/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 11.4552 - mae: 0.8151 - val_loss: 1.0079 - val_mae: 0.6068
Epoch 8/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8.3388 - mae: 0.7586 - val_loss: 1.0078 - val_mae: 0.6073
Epoch 9/50
762/762 ━━━━━━━━━━━━━━━━━━━━ 1s 

2025/11/09 16:24:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 16:24:11 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during keras autologging: BAD_REQUEST: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.IntegrityError) UNIQUE constraint failed: metrics.key, metrics.timestamp, metrics.step, metrics.run_uuid, metrics.value, metrics.is_nan
[SQL: INSERT INTO metrics ("key", value, timestamp, step, is_nan, run_uuid) VALUES (?, ?, ?, ?, ?, ?)]
[parameters: [('loss', 30067.78125, 1762694585167, 0, 0, '71009de2613b46e3bc6f63715dad9dd5'), ('mae', 67.8564224243164, 1762694585167, 0, 0, '71009de2613b46e3bc6f63715dad9dd5'), ('val_loss', 1.0074936151504517, 1762694585167, 0, 0, '71009de2613b46e3bc6f63715dad9dd5'), ('val_mae', 0.6090685129165649, 1762694585167, 0, 0, '71009de2613b46e3bc6f63715dad9dd5')]]
(Background on this error

191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 927us/step
🏃 View run first_run at: http://localhost:5000/#/experiments/1/runs/71009de2613b46e3bc6f63715dad9dd5
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [23]:
second_run()

c:\Users\Aleggg\Documents\code\.venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 14899.4893 - mae: 52.8136 - val_loss: 4.4238 - val_mae: 1.8111
Epoch 2/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 481.7730 - mae: 10.4547 - val_loss: 1.6788 - val_mae: 1.0505
Epoch 3/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 142.3531 - mae: 4.7520 - val_loss: 1.0074 - val_mae: 0.6181
Epoch 4/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 79.8288 - mae: 2.9105 - val_loss: 1.0074 - val_mae: 0.6181
Epoch 5/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 46.0801 - mae: 2.0463 - val_loss: 1.0077 - val_mae: 0.6174
Epoch 6/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 30.6045 - mae: 1.5616 - val_loss: 1.0097 - val_mae: 0.6144
Epoch 7/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 17.7591 - mae: 1.2628 - val_loss: 1.0094 - val_mae: 0.6145
Epoch 8/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 17.6752 - mae: 1.1385 - val_loss: 1.0093 - val_mae: 0.6139
Epoch 9/30
381/381 ━━━━━━━━━━━━━━━━━━━━ 1

2025/11/09 16:24:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
🏃 View run second_run at: http://localhost:5000/#/experiments/1/runs/e034fa451948477bb1bf9779b48a43ff
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [24]:
third_run()

c:\Users\Aleggg\Documents\code\.venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 378569.0625 - mae: 271.4174 - val_loss: 1.3864 - val_mae: 0.9143
Epoch 2/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 10704.0967 - mae: 57.7909 - val_loss: 1.0089 - val_mae: 0.6036
Epoch 3/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 2749.9978 - mae: 22.7506 - val_loss: 1.0091 - val_mae: 0.6031
Epoch 4/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 925.5812 - mae: 9.4276 - val_loss: 1.0094 - val_mae: 0.6025
Epoch 5/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 483.1399 - mae: 5.5067 - val_loss: 1.0095 - val_mae: 0.6022
Epoch 6/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 282.8557 - mae: 3.7004 - val_loss: 1.0095 - val_mae: 0.6023
Epoch 7/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 196.9312 - mae: 2.7781 - val_loss: 1.0094 - val_mae: 0.6023
Epoch 8/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 104.6070 - mae: 1.9053 - val_loss: 1.0095 - val_mae: 0.6022
Epoch 9/100
381/381 ━━

2025/11/09 16:28:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 16:28:09 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during keras autologging: BAD_REQUEST: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.IntegrityError) UNIQUE constraint failed: metrics.key, metrics.timestamp, metrics.step, metrics.run_uuid, metrics.value, metrics.is_nan
[SQL: INSERT INTO metrics ("key", value, timestamp, step, is_nan, run_uuid) VALUES (?, ?, ?, ?, ?, ?)]
[parameters: [('validation_mae', 0.9143163561820984, 1762694706156, 0, 0, '1b283057887e49bf9c658883e830e165'), ('validation_mae', 0.6035512685775757, 1762694707667, 0, 0, '1b283057887e49bf9c658883e830e165'), ('validation_mae', 0.6030839085578918, 1762694709263, 0, 0, '1b283057887e49bf9c658883e830e165'), ('validation_mae', 0.602455735206604, 1762694711049, 0, 0, '1b283057887e49bf9c65

191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
🏃 View run third_run at: http://localhost:5000/#/experiments/1/runs/1b283057887e49bf9c658883e830e165
🧪 View experiment at: http://localhost:5000/#/experiments/1
